In [1]:
# notebook: 04_wdi_macro_extraction.ipynb
# ============================================================================
# CMVTS Extension — WDI macro-indicator extraction (9 countries × 2 vintages)
# ----------------------------------------------------------------------------
# Pulls the OPEN, reproducible macro indicators designed in
# CMVTS_macro_indicator_design.md via the World Bank WDI public API.
# Findex-derived indicators (A4,A5,B1-B6,C1-C5) are NOT fetched here — those come
# from the Findex microdata you already have (notebook 03 pipeline). This file
# covers the WDI-only indicators: A1-A3, A6(proxy), B7, B8, D1-D5.
#
# Output: a tidy country × indicator × year matrix, plus a predictor/outcome tag
# so the circularity-separation rule (design doc §4) is enforced downstream.
#
# Runs locally (this env blocks worldbank.org). No API key needed. If your
# network also blocks it, use the wbgapi fallback (pip install wbgapi) at bottom.
# ============================================================================

import time
import json
import urllib.request
import pandas as pd

# ----------------------------------------------------------------------------
# 0. CONFIG
# ----------------------------------------------------------------------------
# ISO3 codes. Korea = source. Note WB uses 'KOR'.
COUNTRIES = {
    "KOR": "Korea, Rep.",     # SOURCE
    "IDN": "Indonesia",
    "THA": "Thailand",
    "VNM": "Viet Nam",
    "PHL": "Philippines",
    "BGD": "Bangladesh",
    "KHM": "Cambodia",
    "NPL": "Nepal",
    "PAK": "Pakistan",
    "LAO": "Lao PDR",
}

# WDI-only indicators from the design doc. (Findex/GSMA ones handled separately.)
# tag: 'X' = allowed in predictor; 'Y_linked' = feeds outcome, EXCLUDE from predictor;
#      'scale' = macro structure / normalization.
WDI_INDICATORS = {
    # A — digital infrastructure  (predictor-eligible)
    "IT.NET.USER.ZS":   ("A1_internet_use_pct",        "X"),
    "IT.CEL.SETS.P2":   ("A2_mobile_subs_p100",        "X"),
    "IT.NET.BBND.P2":   ("A3_fixed_bbnd_p100",         "X"),
    "IT.NET.SECR.P6":   ("A6_secure_servers_p1m",      "X"),   # AMBS proxy if AMBS absent
    # B — financial access structure (predictor-eligible structural ones)
    "FB.CBK.BRCH.P5":   ("B7_bank_branches_p100k",     "X"),
    "FB.ATM.TOTL.P5":   ("B8_atm_p100k",               "X"),
    # D — macro scale/structure (normalization; predictor-eligible)
    "NY.GNP.PCAP.CD":   ("D1_gni_pc_atlas",            "scale"),
    "SP.URB.TOTL.IN.ZS":("D2_urban_pct",               "X"),
    "SL.TLF.CACT.ZS":   ("D3_labor_participation_pct", "X"),
    "FS.AST.PRVT.GD.ZS":("D4_domestic_credit_priv_gdp","X"),
    "SE.ADT.LITR.ZS":   ("D5_adult_literacy_pct",      "X"),
}

YEARS = "2021:2024"          # fetch the window; we pick 2021 and latest<=2024 per country
API = "https://api.worldbank.org/v2/country/{iso}/indicator/{code}?format=json&date={yrs}&per_page=100"

# ----------------------------------------------------------------------------
# 1. FETCH
# ----------------------------------------------------------------------------
def fetch(iso, code, retries=3):
    url = API.format(iso=iso, code=code, yrs=YEARS)
    for a in range(retries):
        try:
            with urllib.request.urlopen(url, timeout=30) as r:
                d = json.load(r)
            if not isinstance(d, list) or len(d) < 2 or d[1] is None:
                return []
            return [{"iso": iso, "code": code, "year": int(x["date"]),
                     "value": x["value"]} for x in d[1] if x["value"] is not None]
        except Exception as e:
            if a == retries - 1:
                print(f"  [warn] {iso}/{code}: {type(e).__name__}")
            time.sleep(1.5)
    return []

print("Fetching WDI indicators for", len(COUNTRIES), "countries...")
records = []
for iso in COUNTRIES:
    for code in WDI_INDICATORS:
        records.extend(fetch(iso, code))
    print("  done:", iso)
    time.sleep(0.3)

raw = pd.DataFrame(records)
print("\nRaw records:", len(raw))

# ----------------------------------------------------------------------------
# 2. PICK 2021 + latest available <=2024 per (country, indicator)
# ----------------------------------------------------------------------------
def pick_vintages(g):
    v2021 = g.loc[g.year == 2021, "value"]
    latest = g.loc[g.year <= 2024].sort_values("year")
    v_latest = latest["value"].iloc[-1] if len(latest) else None
    y_latest = latest["year"].iloc[-1] if len(latest) else None
    return pd.Series({"v_2021": v2021.iloc[0] if len(v2021) else None,
                      "v_latest": v_latest, "year_latest": y_latest})

vint = (raw.groupby(["iso", "code"]).apply(pick_vintages, include_groups=False)
            .reset_index())
vint["indicator"] = vint["code"].map(lambda c: WDI_INDICATORS[c][0])
vint["tag"]       = vint["code"].map(lambda c: WDI_INDICATORS[c][1])
vint["economy"]   = vint["iso"].map(COUNTRIES)

# ----------------------------------------------------------------------------
# 3. WIDE MATRICES (one per vintage) + coverage report
# ----------------------------------------------------------------------------
def wide(col):
    m = vint.pivot(index="economy", columns="indicator", values=col)
    return m.reindex(list(COUNTRIES.values()))

mat_2021   = wide("v_2021")
mat_latest = wide("v_latest")

print("\n=== 2021 vintage matrix ===")
print(mat_2021.round(1).to_string())
print("\n=== latest(<=2024) vintage matrix ===")
print(mat_latest.round(1).to_string())

# coverage: how many countries have each indicator
cov = pd.DataFrame({
    "cov_2021":   mat_2021.notna().sum(),
    "cov_latest": mat_latest.notna().sum(),
    "n_countries": len(COUNTRIES),
})
print("\n=== Coverage (non-missing countries per indicator) ===")
print(cov.to_string())

# flag indicators with weak coverage (design doc warned: D5 literacy may be sparse)
weak = cov[cov["cov_latest"] < len(COUNTRIES)].index.tolist()
if weak:
    print("\n[note] weak-coverage indicators (consider carry-forward or drop):", weak)

# ----------------------------------------------------------------------------
# 4. SAVE + predictor/outcome separation tag
# ----------------------------------------------------------------------------
mat_2021.to_csv("wdi_macro_2021.csv")
mat_latest.to_csv("wdi_macro_latest.csv")
vint.to_csv("wdi_macro_long.csv", index=False)

pred_cols = [WDI_INDICATORS[c][0] for c in WDI_INDICATORS
             if WDI_INDICATORS[c][1] in ("X", "scale")]
print("\nPredictor-eligible WDI columns (no outcome-linked vars here):")
print("  ", pred_cols)
print("\nSaved: wdi_macro_2021.csv, wdi_macro_latest.csv, wdi_macro_long.csv")
print("Next: merge with Findex-derived A4/A5/B/C indicators, normalize to 0-1,")
print("      then compute C2 (Spearman) and C3 (cosine) for the real CMVTS.")

# ----------------------------------------------------------------------------
# FALLBACK — if urllib is blocked, use the official wbgapi package
# ----------------------------------------------------------------------------
# pip install wbgapi
# import wbgapi as wb
# codes = list(WDI_INDICATORS.keys())
# df = wb.data.DataFrame(codes, list(COUNTRIES.keys()), time=range(2021, 2025),
#                        labels=True).reset_index()
# # then reshape as above

Fetching WDI indicators for 10 countries...
  done: KOR
  done: IDN
  done: THA
  done: VNM
  done: PHL
  done: BGD
  done: KHM
  done: NPL
  done: PAK
  done: LAO

Raw records: 395

=== 2021 vintage matrix ===
indicator    A1_internet_use_pct  A2_mobile_subs_p100  A3_fixed_bbnd_p100  A6_secure_servers_p1m  B7_bank_branches_p100k  B8_atm_p100k  D1_gni_pc_atlas  D2_urban_pct  D3_labor_participation_pct  D4_domestic_credit_priv_gdp  D5_adult_literacy_pct
economy                                                                                                                                                                                                                                              
Korea, Rep.                 97.6                140.5                44.3                 7334.1                    13.6         256.9          37450.0          81.2                        62.6                        159.9                    NaN
Indonesia                   62.1                132